# QKeras Quantization Sweep (QDense Architecture)

This notebook implements **Quantization-Aware Training (QAT)** using QKeras `QDense` layers,
sweeping all 16 combinations of weight and activation bit-widths: {2, 4, 8, 32} × {2, 4, 8, 32}.

# Why QDense (not QConv2D)
QKeras 0.9's `QConv2D` layer has a known incompatibility with TF ≥ 2.13 on Apple Silicon
(Mac M1/M2/M3). Specifically, `KerasTensor` lost its `.node` attribute in TF 2.13, which
QKeras uses internally during model construction. This affects both the Functional API and
Sequential API. The `QConv2D` version of this notebook is preserved in `train_quantized.ipynb`
and should be run on the Fermilab cluster (Linux x86, TF ≤ 2.12) where QKeras works natively.

# Architectures
- **CNN-QDense**: flatten syndrome bits → QDense(128) → QDense(64) → output
- **RCNN-QDense**: reshape → LSTM(32) → QDense(64) → output

# Outputs
- `./results/results_quantization_sweep_qdense.csv`
- Columns: `architecture, w_bits, a_bits, d, p, p_L, model_size_kb, inference_latency_ms`

# References
- Bausch et al. 2023 (arXiv:2310.05900) — transformer decoder for surface codes (related work)
- Coelho et al. 2021 (Nature MI 3:675) — QKeras quantization library
- Perdue, FNAL-QCDecodingTests GitHub repo — original Fermilab codebase this work extends
- Overwater et al. 2022 (IEEE TQE) — hardware cost of NN decoders (closest prior work)

In [6]:
import os
import csv
import time
import tempfile
from datetime import datetime

import numpy as np

# TensorFlow / Keras
# Must use tensorflow-macos==2.15.0 + keras==2.15.0 on Apple Silicon.
# Do NOT install tensorflow-metal — incompatible with this TF version on Mac.
import tensorflow as tf
import keras

# QKeras — quantization-aware training layers
# QDense: quantized fully-connected layer
# QActivation: quantized activation (used between QDense layers)
# quantized_bits(b, i): b = total bits, i = integer bits
#   e.g. quantized_bits(4, 1) = 4-bit fixed-point with 1 integer bit
from qkeras import QDense, QActivation, quantized_bits

print(f'TF version:    {tf.__version__}')   # Expected: 2.15.0
print(f'Keras version: {keras.__version__}') # Expected: 2.15.0
print(f'Imports OK at {datetime.now()}')

TF version:    2.15.0
Keras version: 2.15.0
Imports OK at 2026-06-05 20:38:43.885870


In [2]:
# --- Paths ---
# Datasets were generated in Phase 2 (Stim_generate_datasets.ipynb)
# Each file: data_d{d}_p{p:.3f}_r{ROUNDS}.npz
# Contains: det_evts (detector events), flips (logical observables)
DATA_DIR   = './datasets'
OUTPUT_DIR = './results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Syndrome rounds ---
# r=2 means 2 rounds of stabilizer measurements per sample.
# Syndrome length = (d^2 - 1) * r bits per sample.
ROUNDS = 2

# --- (d, p) configurations ---
# d = code distance (3, 5, 7) — larger d = more physical qubits, better protection
# p = physical error rate — probability of a gate/measurement error
# Surface code threshold is ~0.57% (Fowler et al. 2012),
# so p=0.001 and p=0.005 are below threshold, p=0.01 and p=0.05 above.
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]

# --- Quantization sweep grid ---
# (w_bits, a_bits): weight bit-width × activation bit-width
# 32 = full-precision float32 baseline (no quantization applied)
# 2, 4, 8 = aggressively quantized
# Total: 4 × 4 = 16 configurations per architecture
QUANT_CONFIGS = [
    (2, 2),  (2, 4),  (2, 8),  (2, 32),
    (4, 2),  (4, 4),  (4, 8),  (4, 32),
    (8, 2),  (8, 4),  (8, 8),  (8, 32),
    (32, 2), (32, 4), (32, 8), (32, 32),
]

# --- Training hyperparameters ---
# Same as Phase 4 (train_fullprecision.ipynb) for fair comparison.
# Early stopping monitors val_loss; if no improvement for PATIENCE epochs, stop.
# restore_best_weights=True ensures we keep the best checkpoint.
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
MAX_EPOCHS    = 100
PATIENCE      = 10

# --- Validation mode ---
# VALIDATE_ONLY = True:  run d=5, p=0.01 only (32 runs) to confirm pipeline works
# VALIDATE_ONLY = False: run all 12 (d,p) configs (384 runs) for full sweep
# Always validate first before committing to the full sweep.
VALIDATE_ONLY = True

sweep_configs = [(5, 0.010)] if VALIDATE_ONLY else CONFIGS

print(f'Sweep configs:         {sweep_configs}')
print(f'Quantization configs:  {len(QUANT_CONFIGS)}')
print(f'Architectures:         CNN-QDense, RCNN-QDense')
print(f'Total runs:            {len(sweep_configs) * len(QUANT_CONFIGS) * 2}')
print(f'Validate only:         {VALIDATE_ONLY}')

Sweep configs:         [(5, 0.01)]
Quantization configs:  16
Architectures:         CNN-QDense, RCNN-QDense
Total runs:            32
Validate only:         True


In [3]:
# Cell 3 — Model Definitions

#
# Both architectures use keras.Sequential (not Functional API).
# QKeras 0.9 QDense layers are compatible with Sequential on TF 2.15.
#
# When w_bits=32, plain Keras Dense layers are used (no quantization),
# so (w=32, a=32) serves as the in-sweep full-precision baseline and
# should closely match Phase 4 results.
#
# QAT methodology: quantization constraints are active DURING training,
# not applied post-hoc. The network learns weights that work well under
# the specified bit-width constraints from the first epoch.

def build_qdense_cnn(input_shape, output_shape, w_bits, a_bits):
    """
    Quantized dense network (CNN-QDense).
    Architecture: Input → QDense(128) → QActivation → QDense(64) → QActivation → Dense(output)
    
    The name 'CNN-QDense' reflects that this replaces the CNN architecture
    from Phase 4 with a quantized dense equivalent. The QConv2D version
    (train_quantized.ipynb) is reserved for the Fermilab cluster.

    Args:
        input_shape:  tuple, e.g. (48,) for d=5 r=2
        output_shape: tuple, e.g. (1,) for single logical qubit
        w_bits:       int, weight bit-width (2/4/8/32)
        a_bits:       int, activation bit-width (2/4/8/32)

    Returns:
        Compiled-ready keras.Sequential model
    """
    if w_bits == 32:
        # Full-precision baseline — plain Dense layers, no QKeras
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dense(64,  activation='relu'),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ], name='CNN_QDense_fp32')
    else:
        # Quantized path
        # quantized_bits(w_bits, 1): fixed-point with 1 integer bit and (w_bits-1) fractional bits
        # quantized_relu(a_bits): ReLU clipped and quantized to a_bits
        # The final Dense output layer is NOT quantized — sigmoid output stays float
        w_q = quantized_bits(w_bits, 1)
        a_q = f'quantized_relu({a_bits})'
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            QDense(128, kernel_quantizer=w_q, bias_quantizer=w_q),
            QActivation(a_q),
            QDense(64,  kernel_quantizer=w_q, bias_quantizer=w_q),
            QActivation(a_q),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ], name=f'CNN_QDense_w{w_bits}a{a_bits}')
    return model


def build_qdense_rcnn(input_shape, output_shape, w_bits, a_bits):
    """
    Quantized recurrent network (RCNN-QDense).
    Architecture: Input → Reshape(1, n) → LSTM(32) → QDense(64) → QActivation → Dense(output)

    The LSTM captures temporal correlations across syndrome rounds.
    LSTM weights are kept full-precision (QKeras does not support QLSTM stably);
    only the dense classification head is quantized.

    Args:
        input_shape:  tuple, e.g. (48,) for d=5 r=2
        output_shape: tuple, e.g. (1,) for single logical qubit
        w_bits:       int, weight bit-width (2/4/8/32)
        a_bits:       int, activation bit-width (2/4/8/32)

    Returns:
        Compiled-ready keras.Sequential model
    """
    if w_bits == 32:
        # Full-precision baseline
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((1, input_shape[0])),  # LSTM expects (batch, timesteps, features)
            keras.layers.LSTM(32, activation='relu'),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ], name='RCNN_QDense_fp32')
    else:
        # Quantized path — LSTM stays full-precision, dense head is quantized
        w_q = quantized_bits(w_bits, 1)
        a_q = f'quantized_relu({a_bits})'
        model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.Reshape((1, input_shape[0])),
            keras.layers.LSTM(32, activation='tanh'),   # tanh is standard LSTM activation
            QDense(64, kernel_quantizer=w_q, bias_quantizer=w_q),
            QActivation(a_q),
            keras.layers.Dense(output_shape[0], activation='sigmoid'),
        ], name=f'RCNN_QDense_w{w_bits}a{a_bits}')
    return model


def measure_model_size_kb(model):
    """
    Measure model size in kilobytes by saving to a temp .h5 file and checking disk size.
    This is a proxy for FPGA resource cost — smaller models = fewer LUTs and BRAMs needed.
    """
    with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as f:
        tmp = f.name
    model.save(tmp)
    kb = os.path.getsize(tmp) / 1024
    os.remove(tmp)
    return kb


def measure_inference_latency_ms(model, X_sample, n_repeats=10):
    """
    Measure wall-clock inference latency in milliseconds for a batch of 1000 samples.
    Runs n_repeats times and returns the mean.
    
    One warm-up call is made first to avoid measuring JIT compilation overhead.
    Latency is a direct measure of speedup — lower bit-width models should be faster
    on hardware that supports low-precision arithmetic (e.g. FPGA, edge accelerators).
    """
    batch = X_sample[:1000]
    model.predict(batch, verbose=0)  # warm-up pass
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        model.predict(batch, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


# --- Sanity check: confirm input shapes for all distances ---
# syndrome length = (d^2 - 1) * rounds
# d=3: (9-1)*2 = 16 bits
# d=5: (25-1)*2 = 48 bits
# d=7: (49-1)*2 = 96 bits
print('Input shape sanity check:')
for d in [3, 5, 7]:
    n = (d**2 - 1) * ROUNDS
    print(f'  d={d}: input_shape=({n},)')
print('Model builders ready.')

Input shape sanity check:
  d=3: input_shape=(16,)
  d=5: input_shape=(48,)
  d=7: input_shape=(96,)
Model builders ready.


In [4]:
# Cell 4 — Training Sweep

#
# Outer loop: (d, p) configurations
# Middle loop: (w_bits, a_bits) quantization configurations
# Inner loop: architectures (CNN-QDense, RCNN-QDense)
#
# Results are written to CSV after EACH run so partial results
# are saved even if the sweep is interrupted.
#
# keras.backend.clear_session() is called after each model to
# free memory and prevent layer name collisions across runs.

csv_path   = f'{OUTPUT_DIR}/results_quantization_sweep_qdense.csv'
fieldnames = ['architecture', 'w_bits', 'a_bits', 'd', 'p',
              'p_L', 'model_size_kb', 'inference_latency_ms']

# Write CSV header (overwrites any previous file)
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

results    = []
total_runs = len(sweep_configs) * len(QUANT_CONFIGS) * 2
run_idx    = 0
t_start    = datetime.now()

for d, p in sweep_configs:

    # --- Load dataset ---
    # det_evts: detector events (syndrome bits), shape (N, syndrome_length)
    # flips:    logical observable flips, shape (N, n_observables)
    # Cast to float32 for TF compatibility
    data     = np.load(f'{DATA_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz')
    det_evts = data['det_evts'].astype(np.float32)
    flips    = data['flips'].astype(np.float32)

    # --- 80/10/10 train/val/test split ---
    # Fixed split (no shuffle) — same as Phase 4 for consistency
    n_train, n_val = 800_000, 100_000
    X_train = det_evts[:n_train]
    y_train = flips[:n_train]
    X_val   = det_evts[n_train:n_train+n_val]
    y_val   = flips[n_train:n_train+n_val]
    X_test  = det_evts[n_train+n_val:]
    y_test  = flips[n_train+n_val:]

    input_shape  = X_train.shape[1:]
    output_shape = y_train.shape[1:]

    print(f'\n{"="*65}')
    print(f'd={d} p={p:.3f} | train={X_train.shape} val={X_val.shape} test={X_test.shape}')
    print(f'{"="*65}')

    for w_bits, a_bits in QUANT_CONFIGS:
        for arch_name, build_fn in [('CNN-QDense',  build_qdense_cnn),
                                    ('RCNN-QDense', build_qdense_rcnn)]:
            run_idx += 1
            print(f'\n  [{run_idx}/{total_runs}] {arch_name} w={w_bits} a={a_bits} | d={d} p={p:.3f}')

            try:
                # Build model
                model = build_fn(input_shape, output_shape, w_bits, a_bits)

                # Use legacy Adam optimizer — avoids M1/M2 Mac slowness warning
                # with the default tf.keras.optimizers.Adam on Apple Silicon
                model.compile(
                    optimizer=keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )

                # Train with early stopping
                # EarlyStopping monitors val_loss; stops if no improvement for PATIENCE epochs
                # restore_best_weights=True reverts to best checkpoint before returning
                model.fit(
                    X_train, y_train,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val),
                    epochs=MAX_EPOCHS,
                    callbacks=[keras.callbacks.EarlyStopping(
                        patience=PATIENCE,
                        restore_best_weights=True
                    )],
                    verbose=1
                )

                # --- Evaluate on test set ---
                # p_L = logical error rate = fraction of test samples where
                # the decoder's prediction disagrees with the true logical flip.
                # .any(axis=1) counts a sample as wrong if ANY output bit is wrong
                # (correct for multi-observable codes, safe for single-observable too)
                pred = (model.predict(X_test, verbose=0) > 0.5).astype(float)
                p_L  = float((pred != y_test).any(axis=1).sum()) / len(y_test)

                # Measure model size and inference latency
                kb  = measure_model_size_kb(model)
                lat = measure_inference_latency_ms(model, X_test)

                print(f'  -> p_L={p_L:.6f}  size={kb:.1f}KB  latency={lat:.1f}ms')

                # --- Save result ---
                row = {
                    'architecture':         arch_name,
                    'w_bits':               w_bits,
                    'a_bits':               a_bits,
                    'd':                    d,
                    'p':                    p,
                    'p_L':                  round(p_L, 8),
                    'model_size_kb':        round(kb, 2),
                    'inference_latency_ms': round(lat, 2),
                }
                results.append(row)

                # Append to CSV immediately so partial results survive interruptions
                with open(csv_path, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

            except Exception as e:
                print(f'  ERROR: {e}')

            finally:
                # Free GPU/CPU memory and reset layer name counters between runs
                try:
                    del model
                except NameError:
                    pass
                keras.backend.clear_session()

print(f'\nDone: {len(results)}/{total_runs} runs succeeded.')
print(f'Elapsed: {datetime.now() - t_start}')
print(f'CSV saved to: {csv_path}')


d=5 p=0.010 | train=(800000, 48) val=(100000, 48) test=(100000, 48)

  [1/32] CNN-QDense w=2 a=2 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 9s 3ms/step - loss: 0.2670 - accuracy: 0.8807 - val_loss: 0.2163 - val_accuracy: 0.9041
Epoch 2/100
3125/3125 [==============================] - 9s 3ms/step - loss: 0.1927 - accuracy: 0.9167 - val_loss: 0.1865 - val_accuracy: 0.9196
Epoch 3/100
3125/3125 [==============================] - 9s 3ms/step - loss: 0.1739 - accuracy: 0.9262 - val_loss: 0.1724 - val_accuracy: 0.9266
Epoch 4/100
3125/3125 [==============================] - 9s 3ms/step - loss: 0.1764 - accuracy: 0.9246 - val_loss: 0.1761 - val_accuracy: 0.9229
Epoch 5/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.1792 - accuracy: 0.9235 - val_loss: 0.1694 - val_accuracy: 0.9286
Epoch 6/100
3125/3125 [==============================] - 10s 3ms/step - loss: 0.1758 - accuracy: 0.9256 - val_loss: 0.1673 - val_accuracy: 0.9292
Epoch 7/100
3125/

/opt/anaconda3/envs/qkeras-tf215/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
/opt/anaconda3/envs/qkeras-tf215/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


  -> p_L=0.069040  size=209.3KB  latency=57.2ms

  [2/32] RCNN-QDense w=2 a=2 | d=5 p=0.010
Epoch 1/100
3125/3125 [==============================] - 7s 2ms/step - loss: 0.2466 - accuracy: 0.8930 - val_loss: 0.1697 - val_accuracy: 0.9299
Epoch 2/100
3125/3125 [==============================] - 6s 2ms/step - loss: 0.1550 - accuracy: 0.9366 - val_loss: 0.1446 - val_accuracy: 0.9407
Epoch 3/100
3125/3125 [==============================] - 6s 2ms/step - loss: 0.1368 - accuracy: 0.9456 - val_loss: 0.1297 - val_accuracy: 0.9480
Epoch 4/100
3125/3125 [==============================] - 6s 2ms/step - loss: 0.1255 - accuracy: 0.9513 - val_loss: 0.1213 - val_accuracy: 0.9528
Epoch 5/100
3125/3125 [==============================] - 6s 2ms/step - loss: 0.1180 - accuracy: 0.9548 - val_loss: 0.1124 - val_accuracy: 0.9569
Epoch 6/100
3125/3125 [==============================] - 6s 2ms/step - loss: 0.1117 - accuracy: 0.9575 - val_loss: 0.1099 - val_accuracy: 0.9581
Epoch 7/100
3125/3125 [===============

In [5]:
# Cell 5 — Results Summary

# Reads the CSV and prints:
#   1. Full results table per architecture
#   2. Best config per architecture (lowest p_L)
#   3. Full-precision baseline (w=32, a=32) for comparison
#
# This gives a quick read of the Pareto frontier before
# generating the full plots in Phase 8.

import pandas as pd

df = pd.read_csv(csv_path)
print(f'Total results: {len(df)} runs\n')

# --- Full results table per architecture ---
for arch in df['architecture'].unique():
    print(f'\n--- {arch} ---')
    sub = df[df['architecture'] == arch][
        ['w_bits', 'a_bits', 'p_L', 'model_size_kb', 'inference_latency_ms']
    ].sort_values(['w_bits', 'a_bits'])
    print(sub.to_string(index=False))

# --- Best config per architecture (lowest p_L = best accuracy) ---
print('\n--- Best config per architecture (lowest p_L) ---')
best = df.loc[df.groupby('architecture')['p_L'].idxmin()][
    ['architecture', 'w_bits', 'a_bits', 'p_L', 'model_size_kb', 'inference_latency_ms']
]
print(best.to_string(index=False))

# --- Full-precision baseline for reference ---
# (w=32, a=32) should closely match Phase 4 results_fullprecision.csv
# If there's a large discrepancy, check that the same dataset files are being used
fp = df[(df['w_bits'] == 32) & (df['a_bits'] == 32)]
print('\n--- Full-precision baseline (w=32, a=32) ---')
print(fp[['architecture', 'p_L', 'model_size_kb', 'inference_latency_ms']].to_string(index=False))

# --- Compression summary ---
# For each architecture, show how much smaller the most aggressive quantization
# (w=2, a=2) is compared to full precision (w=32, a=32)
print('\n--- Compression ratio (w=32,a=32 vs w=2,a=2) ---')
for arch in df['architecture'].unique():
    fp_size = df[(df['architecture'] == arch) & (df['w_bits'] == 32) & (df['a_bits'] == 32)]['model_size_kb'].values
    q2_size = df[(df['architecture'] == arch) & (df['w_bits'] == 2)  & (df['a_bits'] == 2) ]['model_size_kb'].values
    if len(fp_size) > 0 and len(q2_size) > 0:
        ratio = fp_size[0] / q2_size[0]
        print(f'  {arch}: {fp_size[0]:.1f}KB → {q2_size[0]:.1f}KB  ({ratio:.1f}× smaller)')

Total results: 32 runs


--- CNN-QDense ---
 w_bits  a_bits     p_L  model_size_kb  inference_latency_ms
      2       2 0.06904         209.29                 57.22
      2       4 0.06697         209.29                 79.76
      2       8 0.06695         209.29                 56.05
      2      32 0.19994         209.29                 72.09
      4       2 0.03097         209.29                 57.80
      4       4 0.03078         209.29                 31.58
      4       8 0.03078         209.29                 31.82
      4      32 0.19994         209.29                 29.84
      8       2 0.03240         209.29                 31.61
      8       4 0.02989         209.29                 33.67
      8       8 0.03145         209.29                 32.10
      8      32 0.19994         209.29                 31.09
     32       2 0.03049         203.62                 27.15
     32       4 0.03050         203.62                 30.73
     32       8 0.02963         203.62   